# 01 · Exploratory Data Analysis — iNaturalist (Insecta)

Professional EDA over the cleaned, 70/15/15-split dataset produced by
`00_data_ready.ipynb`. Covers dataset overview, image statistics, quality
analysis, annotation consistency, and visualizations. Heavy logic is imported
from `src/data_pipeline/eda.py`; figures are written to `data/interim/eda/`.

Methodology & sources: `docs/eda_best_practices.md`.

## Inlined source
Every `src` module below is embedded as an in-notebook module (no `from src` imports). Edit a module cell to change behaviour.

In [ ]:
import os, sys, json, glob
from pathlib import Path
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
_REPO = Path.cwd()
for _c in [_REPO, *_REPO.parents]:
    if (_c / "data").exists() and (_c / "src").exists(): _REPO = _c; break
os.chdir(_REPO)
# ---- src/config.py (inlined; edit here) ----
"""Central configuration for the Bee-A-Hero data pipeline.

All paths are anchored to the repository root (this file lives in ``src/``),
so the pipeline is portable across machines and does not depend on any
absolute/Windows path. Import this module rather than hard-coding paths.
"""

from pathlib import Path

# --- Repository layout -------------------------------------------------------
REPO_ROOT = _REPO

DATA_DIR: Path = REPO_ROOT / "data"
RAW_DIR: Path = DATA_DIR / "raw"
INTERIM_DIR: Path = DATA_DIR / "interim"
PROCESSED_DIR: Path = DATA_DIR / "processed"
BACKUP_DIR: Path = DATA_DIR / "_backup"

INAT_DIR: Path = RAW_DIR / "iNaturist"
FLOWER_DIR: Path = RAW_DIR / "Flower"
# Roboflow bee-detection COCO export (BEE.v8i). Place it here on any machine
# (git-ignored); overridable via the BEE_COCO_DIR env var for other layouts.
BEE_COCO_DIR: Path = Path(__import__("os").environ.get("BEE_COCO_DIR", RAW_DIR / "BEE_coco"))
# Videos to run the visit-counter on.
TEST_VIDEO_DIR: Path = RAW_DIR / "Test_Video"

# Published trained-weights repo on the Hugging Face Hub (for teammates to pull).
HF_WEIGHTS_REPO: str = "Manheim/bee-a-hero-cv"

# iNaturalist splits present on disk. ``public_test`` is unlabeled
# (annotations: 0) and is inference-only — it is NOT part of the labeled split.
INAT_LABELED_SPLITS: tuple[str, ...] = ("train_mini", "val")
INAT_UNLABELED_SPLIT: str = "public_test"

# Generated-artifact locations (git-ignored; see .gitignore data/interim/*).
MANIFEST_DIR: Path = INTERIM_DIR / "manifests"
EDA_DIR: Path = INTERIM_DIR / "eda"
REMOVED_DIR: Path = BACKUP_DIR / "removed"

# --- Reproducibility ---------------------------------------------------------
SEED: int = 42

# --- Split configuration -----------------------------------------------------
# Train/val/test proportions carved from the pooled labeled Insecta images.
SPLIT_RATIOS: dict[str, float] = {"train": 0.70, "val": 0.15, "test": 0.15}

# --- Taxonomy targets --------------------------------------------------------
# Keep only folders whose taxonomic Class == Insecta.
TARGET_CLASS: str = "Insecta"

# Bee families (Hymenoptera) tagged as a subset of interest for the project.
BEE_FAMILIES: frozenset[str] = frozenset({
    "Andrenidae", "Apidae", "Colletidae", "Halictidae",
    "Megachilidae", "Melittidae", "Stenotritidae",
})

# Valid image extensions.
IMAGE_EXTS: frozenset[str] = frozenset({".jpg", ".jpeg", ".png"})
from types import SimpleNamespace
C = SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper()})
REPO = _REPO
print("repo:", REPO)

#### `src/data_pipeline/eda.py` (inlined)

In [ ]:
# ===== src/data_pipeline/eda.py  (inlined module — edit here) =====
import types as _t, sys as _s
eda = _t.ModuleType("eda")
eda.C = C
_src = r'''
"""Reusable EDA & image-quality primitives for the Bee-A-Hero dataset.

These functions are imported by BOTH ``notebooks/00_data_ready.ipynb``
(integrity / quality gate) and ``notebooks/01_eda.ipynb`` (visual analysis), so
the heavy logic lives here once and the notebooks stay presentation-only.

Everything reads the Phase-4 manifest (``data/interim/manifests/split_manifest.csv``)
as the single source of truth for which images exist and their labels.
"""

import multiprocessing
import random
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image


# Large iNaturalist frames are legitimate; disable the decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

# Python 3.14 defaults to the "forkserver" start method, which re-imports the
# entry module in each worker and breaks under stdin/notebook execution. "fork"
# is safe here (workers only do PIL + numpy) and needs no re-import.
try:
    _MP = multiprocessing.get_context("fork")   # Linux/Mac: avoids py3.14 forkserver bug
except ValueError:
    _MP = multiprocessing.get_context()          # Windows: default (spawn)


# --------------------------------------------------------------------------- #
# Data access
# --------------------------------------------------------------------------- #
def load_manifest_df():
    """Return the Phase-4 split manifest as a DataFrame."""
    import pandas as pd
    return pd.read_csv(C.MANIFEST_DIR / "split_manifest.csv")


def manifest_paths(df=None) -> list[str]:
    df = load_manifest_df() if df is None else df
    return df["path"].tolist()


# --------------------------------------------------------------------------- #
# Integrity — corrupted / unreadable images
# --------------------------------------------------------------------------- #
def _check_one(path_str: str):
    try:
        with Image.open(path_str) as im:
            im.verify()               # verify header + payload without full decode
        return None
    except Exception as e:            # unreadable / truncated / corrupt
        return (path_str, repr(e))


def scan_corrupted(paths, workers: int | None = None, chunk: int = 64) -> list[tuple[str, str]]:
    """Return ``[(path, error)]`` for every image that fails to open/verify."""
    paths = [str(p) for p in paths]
    bad: list[tuple[str, str]] = []
    with ProcessPoolExecutor(max_workers=workers, mp_context=_MP) as ex:
        for res in ex.map(_check_one, paths, chunksize=chunk):
            if res is not None:
                bad.append(res)
    return bad


# --------------------------------------------------------------------------- #
# Per-image statistics (sampled — full set is 151k images)
# --------------------------------------------------------------------------- #
def _laplacian_var(gray: np.ndarray) -> float:
    """Variance of the Laplacian — a standard sharpness/blur proxy (numpy-only).

    Low values indicate blur (little high-frequency energy). Equivalent to the
    OpenCV ``cv2.Laplacian(...).var()`` blur score without the cv2 dependency.
    """
    g = gray
    lap = (-4.0 * g[1:-1, 1:-1]
           + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:])
    return float(lap.var()) if lap.size else 0.0


def _meta_one(path_str: str):
    try:
        with Image.open(path_str) as im:
            w, h = im.size
            mode = im.mode
            gray = np.asarray(im.convert("L"), dtype=np.float32)
            brightness = float(gray.mean())
            contrast = float(gray.std())
            blur = _laplacian_var(gray)
            is_gray = mode in ("L", "1")
            if mode == "RGB":
                rgb = np.asarray(im, dtype=np.int16)
                if rgb.ndim == 3 and rgb.shape[2] >= 3:
                    dg = np.abs(rgb[..., 0] - rgb[..., 1]).mean()
                    db = np.abs(rgb[..., 1] - rgb[..., 2]).mean()
                    is_gray = bool(dg < 2 and db < 2)
            blank = bool(contrast < 1.0)
        return (path_str, w, h, mode, brightness, contrast, blur, int(is_gray), int(blank))
    except Exception:
        return (path_str, None, None, None, None, None, None, None, None)


def sample_image_stats(paths, sample: int | None = 6000, seed: int = C.SEED,
                       workers: int | None = None):
    """Compute width/height/aspect/mode/brightness/contrast/grayscale/blank.

    Sampled by default for speed; pass ``sample=None`` to scan everything.
    """
    import pandas as pd
    paths = [str(p) for p in paths]
    rng = random.Random(seed)
    if sample and len(paths) > sample:
        paths = rng.sample(paths, sample)
    rows = []
    with ProcessPoolExecutor(max_workers=workers, mp_context=_MP) as ex:
        for r in ex.map(_meta_one, paths, chunksize=32):
            rows.append(r)
    df = pd.DataFrame(rows, columns=["path", "width", "height", "mode",
                                     "brightness", "contrast", "blur",
                                     "is_gray", "blank"])
    df["aspect"] = df["width"] / df["height"]
    return df


# --------------------------------------------------------------------------- #
# Near-duplicate detection (perceptual hash)
# --------------------------------------------------------------------------- #
def find_near_duplicates(paths, sample: int | None = 8000, seed: int = C.SEED):
    """Group images sharing an identical perceptual hash (resize/re-encode dups).

    Returns a list of path-groups (each length >= 2). Sampled for tractability.
    """
    import imagehash
    rng = random.Random(seed)
    paths = [str(p) for p in paths]
    if sample and len(paths) > sample:
        paths = rng.sample(paths, sample)
    buckets: dict[str, list[str]] = defaultdict(list)
    for p in paths:
        try:
            with Image.open(p) as im:
                buckets[str(imagehash.phash(im))].append(p)
        except Exception:
            continue
    return [ps for ps in buckets.values() if len(ps) > 1]


# --------------------------------------------------------------------------- #
# Aggregations for plots
# --------------------------------------------------------------------------- #
def class_distribution(df=None):
    """Per-class image counts (DataFrame indexed by class_name)."""
    df = load_manifest_df() if df is None else df
    return df.groupby("class_name").size().sort_values(ascending=False)


def split_class_matrix(df=None):
    """Rows = class, columns = split, values = image counts."""
    df = load_manifest_df() if df is None else df
    return df.pivot_table(index="class_name", columns="split",
                          values="path", aggfunc="count", fill_value=0)


def order_distribution(df=None):
    """Image counts per taxonomic order."""
    df = load_manifest_df() if df is None else df
    return df.groupby("order").size().sort_values(ascending=False)


def imbalance_metrics(counts) -> dict:
    """Imbalance ratio + Gini over a series/array of per-class counts."""
    x = np.sort(np.asarray(counts, dtype=float))
    n = len(x)
    ratio = float(x.max() / x.min()) if x.min() > 0 else float("inf")
    if x.sum() == 0:
        gini = 0.0
    else:
        cum = np.cumsum(x)
        gini = float((n + 1 - 2 * (cum / cum[-1]).sum()) / n)
    return {"num_classes": n, "min": float(x.min()), "max": float(x.max()),
            "mean": float(x.mean()), "imbalance_ratio": ratio, "gini": round(gini, 4)}


def sample_grid_paths(df=None, per_split: int = 8, seed: int = C.SEED) -> list[str]:
    """Deterministic sample of image paths for a preview grid."""
    df = load_manifest_df() if df is None else df
    rng = random.Random(seed)
    out: list[str] = []
    for split in ("train", "val", "test"):
        pool = df[df["split"] == split]["path"].tolist()
        out += rng.sample(pool, min(per_split, len(pool)))
    return out
'''
exec(compile(_src, "src/data_pipeline/eda.py", "exec"), eda.__dict__)
_s.modules["eda"] = eda
globals()["eda"] = eda

## 1 · Dataset overview
Image counts, class counts, split distribution, dataset size on disk.

In [ ]:
sizes = df.assign(bytes=[ (REPO/p).stat().st_size for p in paths ])
overview = {
    "images_total": int(len(df)),
    "num_classes": int(df["class_id"].nunique()),
    "num_orders": int(df["order"].nunique()),
    "num_families": int(df["family"].nunique()),
    "bee_images": int(df["is_bee"].sum()),
    "split_counts": df["split"].value_counts().to_dict(),
    "split_fractions": {k: round(v/len(df),4) for k,v in df["split"].value_counts().items()},
    "disk_size_gb": round(sizes["bytes"].sum()/1e9, 2),
    "mean_image_kb": round(sizes["bytes"].mean()/1e3, 1),
}
SUMMARY["overview"] = overview
print(json.dumps(overview, indent=2))

## 2 · Class & order distribution + imbalance

In [ ]:
counts = eda.class_distribution(df)
orders = eda.order_distribution(df)
imb = eda.imbalance_metrics(counts)
SUMMARY["imbalance"] = imb
print("imbalance:", json.dumps(imb, indent=2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(counts.values, bins=range(int(counts.min()), int(counts.max())+2), color="#4C72B0", align="left")
ax[0].set(title="Per-class image counts", xlabel="images / class", ylabel="# classes")
orders.plot.barh(ax=ax[1], color="#55A868"); ax[1].invert_yaxis()
ax[1].set(title=f"Images per taxonomic order (n={len(orders)})", xlabel="images")
SUMMARY["fig_class_order"] = save(fig, "class_and_order_distribution.png")

## 3 · Split distribution, per-class coverage & bee subset

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sc = df["split"].value_counts()
ax[0].pie(sc.values, labels=[f"{k}\n{v:,}" for k,v in sc.items()], autopct="%1.1f%%",
          colors=["#4C72B0","#DD8452","#55A868"]); ax[0].set_title("Split distribution (70/15/15)")

bee = df.groupby(["split","is_bee"]).size().unstack(fill_value=0)
bee.columns = ["non-bee","bee"]
bee.plot.bar(ax=ax[1], stacked=True, color=["#C44E52","#8172B3"]); ax[1].set_title("Bee vs non-bee per split")
ax[1].set_xlabel("split"); ax[1].tick_params(axis="x", rotation=0)

order_split = df.pivot_table(index="order", columns="split", values="path", aggfunc="count", fill_value=0)
sns.heatmap(order_split, ax=ax[2], cmap="viridis", cbar_kws={"label":"images"})
ax[2].set_title("Order × split heatmap")
SUMMARY["fig_split"] = save(fig, "split_bee_order_heatmap.png")

# every class present in every split?
cov = (df.pivot_table(index="class_name", columns="split", values="path", aggfunc="count", fill_value=0) > 0)
SUMMARY["every_class_all_splits"] = bool(cov.all().all())
print("every class in all splits:", SUMMARY["every_class_all_splits"])

## 4 · Image statistics (sampled)
Resolution, aspect ratio, brightness, contrast, blur, colour mode. Sampled for speed (representative of 151k).

In [ ]:
stats = eda.sample_image_stats(paths, sample=12000, workers=os.cpu_count())
stats = stats.dropna(subset=["width"])
res = {"width_min_med_max":[int(stats.width.min()),int(stats.width.median()),int(stats.width.max())],
       "height_min_med_max":[int(stats.height.min()),int(stats.height.median()),int(stats.height.max())],
       "aspect_med": round(float(stats.aspect.median()),3),
       "brightness_med": round(float(stats.brightness.median()),1),
       "modes": stats["mode"].value_counts().to_dict()}
SUMMARY["image_stats_sampled"] = res
print(json.dumps(res, indent=2))

fig, ax = plt.subplots(2, 3, figsize=(15, 8))
ax[0,0].hist(stats.width, bins=40, color="#4C72B0"); ax[0,0].set_title("Width (px)")
ax[0,1].hist(stats.height, bins=40, color="#4C72B0"); ax[0,1].set_title("Height (px)")
ax[0,2].scatter(stats.width, stats.height, s=4, alpha=.2, color="#4C72B0"); ax[0,2].set(title="Resolution scatter", xlabel="w", ylabel="h")
ax[1,0].boxplot(stats.aspect.dropna(), vert=True); ax[1,0].set_title("Aspect ratio (w/h)")
ax[1,1].hist(stats.brightness, bins=40, color="#DD8452"); ax[1,1].set_title("Brightness (mean gray)")
ax[1,2].hist(np.log1p(stats.blur), bins=40, color="#55A868"); ax[1,2].set_title("Blur: log(1+var(Laplacian))")
SUMMARY["fig_imgstats"] = save(fig, "image_statistics.png")

## 5 · Quality analysis
Grayscale, blank, blurry, exposure outliers, near-duplicates. (Corrupted = 0, verified in `00_data_ready`.)

In [ ]:
blur_thr = float(np.percentile(stats.blur, 5))     # bottom 5% as "blurry" flag
quality = {
    "grayscale_images_sampled": int(stats.is_gray.sum()),
    "blank_images_sampled": int(stats.blank.sum()),
    "blurry_flag_sampled(<=p5)": int((stats.blur <= blur_thr).sum()),
    "very_dark_sampled(bright<30)": int((stats.brightness < 30).sum()),
    "very_bright_sampled(bright>225)": int((stats.brightness > 225).sum()),
    "sample_size": int(len(stats)),
}
near = eda.find_near_duplicates(paths, sample=8000)
quality["near_duplicate_groups_sampled"] = len(near)
SUMMARY["quality"] = quality
print(json.dumps(quality, indent=2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
stats["mode"].value_counts().plot.bar(ax=ax[0], color="#8172B3"); ax[0].set_title("Colour mode"); ax[0].tick_params(axis="x", rotation=0)
ax[1].hist(stats.contrast, bins=40, color="#C44E52"); ax[1].set_title("Contrast (gray std)")
SUMMARY["fig_quality"] = save(fig, "quality_mode_contrast.png")

## 6 · Sample images
Deterministic random examples across splits.

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
grid = eda.sample_grid_paths(df, per_split=8)
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for axx, p in zip(axes.ravel(), grid):
    try:
        axx.imshow(Image.open(REPO/p).convert("RGB"));
    except Exception: pass
    axx.axis("off")
    axx.set_title(Path(p).parts[-2].split("_")[-1][:12], fontsize=7)
fig.suptitle("Random samples — rows: train / val / test", y=1.02)
SUMMARY["fig_samples"] = save(fig, "sample_grid.png")

## 7 · EDA summary

In [ ]:
(EDA / "eda_summary.json").write_text(json.dumps(SUMMARY, indent=2))
print(json.dumps({k:v for k,v in SUMMARY.items() if not str(k).startswith("fig_")}, indent=2))
print("\nfigures in", EDA)